# Simple Fraud Agent Evaluation

This notebook evaluates the simple E2B fraud agent with an AML Investigation-style workflow:
- define a small evaluation dataset
- run the agent case-by-case
- score outputs with deterministic rubric checks
- summarize aggregate performance

In [1]:
%pip install -q aieng-agents python-dotenv pandas langfuse

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import asyncio
import json
import statistics

import pandas as pd
from dotenv import load_dotenv
from aieng.agent_evals.async_client_manager import AsyncClientManager
from aieng.agent_evals.langfuse import init_tracing, is_tracing_enabled

load_dotenv()

LANGFUSE_EXPERIMENT_NAME = "fraud-agent-simple-e2b-evaluation"
LANGFUSE_TRACING_ENABLED = init_tracing(service_name=LANGFUSE_EXPERIMENT_NAME)
langfuse_client = AsyncClientManager.get_instance().langfuse_client

NOTEBOOK_CANDIDATES = [
    Path("/home/coder/eval-agents/implementations/cibc-three/Fraud_Agent_Simple_E2B.ipynb"),
    Path("implementations/cibc-three/Fraud_Agent_Simple_E2B.ipynb"),
    Path("Fraud_Agent_Simple_E2B.ipynb"),
]

SOURCE_NOTEBOOK = next((path for path in NOTEBOOK_CANDIDATES if path.exists()), None)
if SOURCE_NOTEBOOK is None:
    tried = [str(path) for path in NOTEBOOK_CANDIDATES]
    raise FileNotFoundError(f"Could not locate source notebook. Tried: {tried}")

print(f"Source notebook: {SOURCE_NOTEBOOK}")
print(f"Langfuse tracing enabled: {LANGFUSE_TRACING_ENABLED and is_tracing_enabled()}")

2026-03-26 16:57:27,297 WARNING opentelemetry.trace: Overriding of current TracerProvider is not allowed
2026-03-26 16:57:32,269 INFO aieng.agent_evals.langfuse: Langfuse tracing initialized successfully (endpoint: https://us.cloud.langfuse.com/api/public/otel)


Source notebook: /home/coder/eval-agents/implementations/cibc-three/Fraud_Agent_Simple_E2B.ipynb
Langfuse tracing enabled: True


## Load the Agent Notebook

This cell executes the source notebook code cells in-process so the evaluation notebook can call `ask_simple_fraud_agent` directly.

In [3]:
def _extract_python_from_notebook(notebook_path: Path) -> str:
    payload = json.loads(notebook_path.read_text())
    code_chunks = []
    for cell in payload.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", [])
        lines = []
        for line in source:
            if line.lstrip().startswith("%"):
                continue
            lines.append(line)
        if lines:
            code_chunks.append("\n".join(lines))
    return "\n\n".join(code_chunks)


def _strip_top_level_runtime_blocks(code: str) -> str:
    lines = code.splitlines()
    cleaned_lines = []
    skip_block = False
    block_indent = 0

    blocked_prefixes = (
        "simple_session = await simple_session_service.create_session(",
        "response = await ask_simple_fraud_agent(",
    )
    blocked_single_lines = (
        'print(response)',
        'print("Simple fraud agent ready.")',
        'print("Use: await ask_simple_fraud_agent(\'Find suspicious patterns in the database\')")',
    )

    for line in lines:
        stripped = line.lstrip()
        indent = len(line) - len(stripped)

        if skip_block:
            if stripped and indent <= block_indent and not stripped.startswith(")"):
                skip_block = False
            else:
                continue

        if any(stripped.startswith(prefix) for prefix in blocked_prefixes):
            cleaned_lines.append(f"# disabled during eval notebook loading: {stripped}")
            skip_block = True
            block_indent = indent
            continue

        if indent == 0 and stripped in blocked_single_lines:
            cleaned_lines.append(f"# disabled during eval notebook loading: {stripped}")
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


agent_globals = {}
agent_code = _extract_python_from_notebook(SOURCE_NOTEBOOK)
agent_code = _strip_top_level_runtime_blocks(agent_code)

exec(agent_code, agent_globals)

simple_session_service = agent_globals["simple_session_service"]
simple_fraud_agent = agent_globals["simple_fraud_agent"]
simple_session = await simple_session_service.create_session(
    app_name=simple_fraud_agent.name,
    user_id="notebook-user",
    state={},
)
agent_globals["simple_session"] = simple_session

ask_simple_fraud_agent = agent_globals["ask_simple_fraud_agent"]

print("Simple fraud agent loaded for evaluation.")

2026-03-26 16:57:36,092 INFO root: Langfuse host: https://us.cloud.langfuse.com
2026-03-26 16:57:36,100 WARNING opentelemetry.trace: Overriding of current TracerProvider is not allowed
2026-03-26 16:57:36,115 WARNING opentelemetry.trace: Overriding of current TracerProvider is not allowed


Langfuse tracing enabled: fraud-agent-simple-e2b-march25
Simple E2B fraud agent config ready.
Read-only SQL + simple fraud tools ready.
Simple fraud agent loaded for evaluation.


In [4]:
orig_run_readonly_query = agent_globals["run_readonly_query"]

async def debug_run_readonly_query(query, row_limit):
    try:
        return await orig_run_readonly_query(query, row_limit)
    except Exception as exc:
        print("\n--- Rejected Query ---\n", query, "\n----------------------")
        raise

agent_globals["run_readonly_query"] = debug_run_readonly_query
print("Debug wrapper installed for run_readonly_query.")

Debug wrapper installed for run_readonly_query.


## Evaluation Dataset

These cases follow the AML evaluation idea of having structured prompts plus expected signal categories for deterministic checks.

In [5]:
eval_cases = [
    {
        "id": "fraud-001",
        "prompt": (
            "Identify the top 3 highest-risk clients in the database by combining transaction amount "
            "and velocity signals. For each client, state their total transaction value, transaction count, "
            "and a specific reason why the combination of both signals makes them suspicious. "
            "Include at least 2 sample transaction IDs per client. "
            "Rank them explicitly from highest to lowest risk."
        ),
        "expected_signals": ["amount", "velocity", "risk", "recommendations"],
    },
    {
        "id": "fraud-002",
        "prompt": (
            "Find merchants that appear in more than one fraud signal (e.g. both high transaction volume "
            "and broad client/state exposure). For each merchant you identify, name the specific signals "
            "they appear in, provide the supporting counts or amounts, and explain why overlap across "
            "signals increases suspicion compared to a single-signal finding. "
            "Include at least 2 sample transaction IDs per merchant."
        ),
        "expected_signals": ["merchant", "amount", "velocity", "risk", "recommendations"],
    },
    {
        "id": "fraud-003",
        "prompt": (
            "Compare the risk profile of high-velocity clients versus high-amount clients. "
            "Which group poses a higher systemic fraud risk and why? Support your conclusion with "
            "specific counts, amounts, or thresholds drawn from the database — do not make general claims "
            "without citing actual figures. Include sample transaction IDs from each group to justify "
            "the comparison. End with a prioritised list of recommended actions."
        ),
        "expected_signals": ["amount", "velocity", "risk", "recommendations"],
    },
    {
        "id": "fraud-004",
        "prompt": (
            "Write a fraud investigation brief covering all three signals (high-value transactions, "
            "velocity anomalies, and risky merchants). For each signal, identify the single most suspicious "
            "entity (client or merchant), state the specific metric that makes it suspicious (e.g. exact "
            "amount, count, number of states), and include sample transaction IDs tied to each signal. "
            "Cross-reference where the same entity appears in more than one signal. "
            "Conclude with a risk-ranked action plan."
        ),
        "expected_signals": ["amount", "velocity", "merchant", "risk", "recommendations"],
    },
    {
        "id": "fraud-005",
        "prompt": (
            "Which clients or merchants identified as high-risk could plausibly have a legitimate "
            "business explanation for their activity? For each one you name, state the data that "
            "initially flagged them, include sample transaction IDs, describe what a benign explanation "
            "might look like, and explain what additional evidence would be needed to confirm or rule out fraud. "
            "Focus on nuance — do not simply list all flagged entities as suspicious."
        ),
        "expected_signals": ["amount", "velocity", "merchant", "risk", "recommendations"],
    },
]

In [6]:
async def load_fraud_label_ground_truth(limit: int = 10000) -> dict:
    code = f"""
import json
import sqlite3

conn = sqlite3.connect({agent_globals['E2B_SQLITE_PATH']!r})
conn.row_factory = sqlite3.Row

rows = [dict(row) for row in conn.execute(
    "SELECT transaction_id, fraud FROM fraud_labels LIMIT ?",
    ({int(limit)},),
).fetchall()]

payload = {{
    "rows": rows,
}}
print(json.dumps(payload, default=str))
conn.close()
"""
    raw_result = await agent_globals["shared_code_interpreter"].run_code(code=code)
    return json.loads(agent_globals["_extract_stdout_text"](raw_result).strip())


def _is_positive_fraud_label(value) -> bool:
    normalized = str(value).strip().lower()
    return normalized in {"1", "true", "yes", "fraud", "fraudulent", "positive"}


fraud_label_ground_truth = await load_fraud_label_ground_truth()
positive_fraud_transaction_ids = {
    int(row["transaction_id"])
    for row in fraud_label_ground_truth["rows"]
    if row.get("transaction_id") is not None and _is_positive_fraud_label(row.get("fraud"))
}

for case in eval_cases:
    case["ground_truth_positive_transaction_ids"] = sorted(positive_fraud_transaction_ids)
    case["ground_truth_positive_count"] = len(positive_fraud_transaction_ids)

pd.DataFrame(eval_cases)

2026-03-26 16:57:36,211 INFO e2b.api: Request POST https://api.e2b.app/sandboxes
2026-03-26 16:57:36,213 INFO e2b.api.client_async: Request: POST https://api.e2b.app/sandboxes


2026-03-26 16:57:36,582 INFO e2b.api.client_async: Response: 201 https://api.e2b.app/sandboxes
2026-03-26 16:57:36,583 INFO e2b.api: Response 201
2026-03-26 16:57:36,586 INFO e2b.api.client_async: Request: POST https://49999-ihoipe1ecjrolukcmsekf.e2b.app/execute
2026-03-26 16:57:36,711 INFO e2b.api.client_async: Response: 200 https://49999-ihoipe1ecjrolukcmsekf.e2b.app/execute
2026-03-26 16:57:37,167 INFO e2b.api: Request DELETE https://api.e2b.app/sandboxes/ihoipe1ecjrolukcmsekf
2026-03-26 16:57:37,168 INFO e2b.api.client_async: Request: DELETE https://api.e2b.app/sandboxes/ihoipe1ecjrolukcmsekf
2026-03-26 16:57:37,288 INFO e2b.api.client_async: Response: 204 https://api.e2b.app/sandboxes/ihoipe1ecjrolukcmsekf
2026-03-26 16:57:37,290 INFO e2b.api: Response 204


,id,prompt,expected_signals,ground_truth_positive_transaction_ids,ground_truth_positive_count
0,fraud-001,Identify the top 3 highest-risk clients in the...,"[amount, velocity, risk, recommendations]","[7760583, 7803991, 7849591, 7959875, 8049815, ...",14
1,fraud-002,Find merchants that appear in more than one fr...,"[merchant, amount, velocity, risk, recommendat...","[7760583, 7803991, 7849591, 7959875, 8049815, ...",14
2,fraud-003,Compare the risk profile of high-velocity clie...,"[amount, velocity, risk, recommendations]","[7760583, 7803991, 7849591, 7959875, 8049815, ...",14
3,fraud-004,Write a fraud investigation brief covering all...,"[amount, velocity, merchant, risk, recommendat...","[7760583, 7803991, 7849591, 7959875, 8049815, ...",14
4,fraud-005,Which clients or merchants identified as high-...,"[amount, velocity, merchant, risk, recommendat...","[7760583, 7803991, 7849591, 7959875, 8049815, ...",14


## AML-Style Deterministic Graders

These checks are simpler than the AML repo graders, but they follow the same pattern: case-level scoring first, then aggregate run-level metrics.

In [ ]:
import re
import statistics

SECTION_HEADERS = ["ANALYSIS:", "FINDINGS:", "RISK:", "RECOMMENDATIONS:"]
SIGNAL_KEYWORDS = {
    "amount": ["amount", "high-value", "transaction", "outlier"],
    "velocity": ["velocity", "transaction count", "rapid", "frequency", "client"],
    "merchant": ["merchant", "state", "clients", "exposure"],
    "risk": ["risk", "high", "medium", "low", "critical"],
    "recommendations": ["recommend", "review", "monitor", "investigate", "escalate"],
}

# Require at least this many transaction IDs in each response.
MIN_TRANSACTION_IDS_PER_CASE = 3


def score_structure(text: str) -> float:
    return sum(header in text for header in SECTION_HEADERS) / len(SECTION_HEADERS)


def score_expected_signals(text: str, expected_signals: list[str]) -> float:
    lower_text = text.lower()
    hits = 0
    for signal in expected_signals:
        keywords = SIGNAL_KEYWORDS[signal]
        if any(keyword in lower_text for keyword in keywords):
            hits += 1
    return hits / len(expected_signals) if expected_signals else 0.0


def score_evidence_density(text: str) -> float:
    markers = ["transaction", "client", "merchant", "amount", "state"]
    lower_text = text.lower()
    hits = sum(marker in lower_text for marker in markers)
    return min(1.0, hits / 4)


def extract_transaction_ids(text: str) -> set[int]:
    ids: set[int] = set()

    # Capture list-style formats like:
    # "Sample transaction IDs: 7475385, 7475440, 7475633"
    list_matches = re.findall(
        r"(?:sample\s+)?transaction[_\s-]?ids?\s*[:#-]?\s*((?:\d+[\s,;]*(?:and\s+)?)+)",
        text,
        flags=re.IGNORECASE,
    )
    for match in list_matches:
        ids.update(int(m) for m in re.findall(r"\d+", match))

    # Capture single inline formats like:
    # "transaction_id: 12345"
    single_matches = re.findall(
        r"(?:transaction[_\s-]?id\s*[:#-]?\s*)(\d+)",
        text,
        flags=re.IGNORECASE,
    )
    ids.update(int(m) for m in single_matches)

    return ids


def score_transaction_id_coverage(output_text: str, min_ids: int = MIN_TRANSACTION_IDS_PER_CASE) -> float:
    predicted_ids = extract_transaction_ids(output_text)
    if min_ids <= 0:
        return 1.0
    return min(1.0, len(predicted_ids) / min_ids)


def score_ground_truth_alignment(case: dict, output_text: str) -> float:
    predicted_ids = extract_transaction_ids(output_text)
    truth_ids = set(case.get("ground_truth_positive_transaction_ids", []))
    if not truth_ids:
        return 0.0
    if not predicted_ids:
        return 0.0

    true_positives = len(predicted_ids & truth_ids)
    precision = true_positives / len(predicted_ids) if predicted_ids else 0.0
    recall = true_positives / len(truth_ids) if truth_ids else 0.0
    if precision + recall == 0:
        return 0.0
    f1 = (2 * precision * recall) / (precision + recall)
    return min(1.0, round(f1, 3))


def item_level_deterministic_grader(case: dict, output_text: str) -> dict:
    structure_score = score_structure(output_text)
    signals_score = score_expected_signals(output_text, case["expected_signals"])
    evidence_score = score_evidence_density(output_text)
    id_coverage_score = score_transaction_id_coverage(output_text)
    ground_truth_score = score_ground_truth_alignment(case, output_text)

    overall_score = round(
        statistics.mean(
            [structure_score, signals_score, evidence_score, id_coverage_score, ground_truth_score]
        ),
        3,
    )

    predicted_ids = extract_transaction_ids(output_text)

    return {
        "structure_score": round(structure_score, 3),
        "signals_score": round(signals_score, 3),
        "evidence_score": round(evidence_score, 3),
        "id_coverage_score": round(id_coverage_score, 3),
        "predicted_transaction_id_count": len(predicted_ids),
        "min_required_transaction_ids": MIN_TRANSACTION_IDS_PER_CASE,
        "meets_min_transaction_ids": len(predicted_ids) >= MIN_TRANSACTION_IDS_PER_CASE,
        "ground_truth_score": round(ground_truth_score, 3),
        "overall_score": overall_score,
        "pass": overall_score >= 0.7,
    }


def run_level_grader(rows: list[dict]) -> dict:
    overall_scores = [row["overall_score"] for row in rows]
    pass_rate = sum(row["pass"] for row in rows) / len(rows) if rows else 0.0
    ground_truth_scores = [row.get("ground_truth_score", 0.0) for row in rows]
    id_coverage_scores = [row.get("id_coverage_score", 0.0) for row in rows]
    id_requirement_pass_rate = (
        sum(row.get("meets_min_transaction_ids", False) for row in rows) / len(rows)
        if rows
        else 0.0
    )

    return {
        "cases": len(rows),
        "mean_overall_score": round(statistics.mean(overall_scores), 3) if overall_scores else 0.0,
        "min_overall_score": round(min(overall_scores), 3) if overall_scores else 0.0,
        "max_overall_score": round(max(overall_scores), 3) if overall_scores else 0.0,
        "pass_rate": round(pass_rate, 3),
        "mean_ground_truth_score": round(statistics.mean(ground_truth_scores), 3) if ground_truth_scores else 0.0,
        "mean_id_coverage_score": round(statistics.mean(id_coverage_scores), 3) if id_coverage_scores else 0.0,
        "id_requirement_pass_rate": round(id_requirement_pass_rate, 3),
    }

## Run the Evaluation

In [22]:
async def run_eval_case(case: dict) -> dict:
    try:
        output_text = await ask_simple_fraud_agent(case["prompt"], fresh_session=True)
        grades = item_level_deterministic_grader(case, output_text)
        result = {
            "id": case["id"],
            "prompt": case["prompt"],
            "expected_signals": ", ".join(case["expected_signals"]),
            "ground_truth_label_column": case.get("ground_truth_label_column"),
            "ground_truth_positive_count": case.get("ground_truth_positive_count"),
            "output": output_text,
            **grades,
        }
        return result
    except Exception as exc:
        print(f"Error running case {case['id']}: {exc}")
        raise

In [30]:
results = []
for case in eval_cases:
    print(f"Running {case['id']} ...")
    results.append(await run_eval_case(case))

/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/adk/runners.py:1481: DeprecationWarning: deprecated
  save_input_blobs_as_artifacts=run_config.save_input_blobs_as_artifacts,
2026-03-26 17:59:57,721 WARNING google_genai._api_client: Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
2026-03-26 17:59:57,833 INFO google_adk.google.adk.models.google_llm: Sending out request, model: gemini-3-pro-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False


Running fraud-001 ...


/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/genai/_api_client.py:755: DeprecationWarning: Inheritance class AiohttpClientSession from ClientSession is discouraged
  class AiohttpClientSession(aiohttp.ClientSession):  # type: ignore[misc]
2026-03-26 18:00:01,449 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-26 18:00:01,454 INFO e2b.api: Request POST https://api.e2b.app/sandboxes
2026-03-26 18:00:01,455 INFO e2b.api.client_async: Request: POST https://api.e2b.app/sandboxes
2026-03-26 18:00:01,643 INFO e2b.api.client_async: Response: 201 https://api.e2b.app/sandboxes
2026-03-26 18:00:01,645 INFO e2b.api: Response 201
2026-03-26 18:00:01,647 INFO e2b.api.client_async: Request: POST https://49999-iqc3tm7i0olh76ojsurz5.e2b.app/execute
2026-03-26 18:00:01,779 INFO e2b.api.client_async: Response: 200 https://49999-iqc3tm7i0olh76ojsurz5.e2b.app/execute
2026-03-26 18:00:01,928 INFO e2b.api: Request DELETE https://api.e2b.app/sandbox

Running fraud-002 ...


2026-03-26 18:02:44,707 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-26 18:02:44,713 INFO e2b.api: Request POST https://api.e2b.app/sandboxes
2026-03-26 18:02:44,714 INFO e2b.api.client_async: Request: POST https://api.e2b.app/sandboxes
2026-03-26 18:02:44,873 INFO e2b.api.client_async: Response: 201 https://api.e2b.app/sandboxes
2026-03-26 18:02:44,874 INFO e2b.api: Response 201
2026-03-26 18:02:44,877 INFO e2b.api.client_async: Request: POST https://49999-ihl3lm1003sqv24fvv22p.e2b.app/execute
2026-03-26 18:02:45,003 INFO e2b.api.client_async: Response: 200 https://49999-ihl3lm1003sqv24fvv22p.e2b.app/execute
2026-03-26 18:02:45,117 INFO e2b.api: Request DELETE https://api.e2b.app/sandboxes/ihl3lm1003sqv24fvv22p
2026-03-26 18:02:45,118 INFO e2b.api.client_async: Request: DELETE https://api.e2b.app/sandboxes/ihl3lm1003sqv24fvv22p
2026-03-26 18:02:45,234 INFO e2b.api.client_async: Response: 204 https://api.e2b.app/sandboxes/ihl3lm1003sqv24fvv22p

...

2026-03-26 18:04:15,205 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-26 18:04:15,211 INFO e2b.api: Request POST https://api.e2b.app/sandboxes
2026-03-26 18:04:15,212 INFO e2b.api.client_async: Request: POST https://api.e2b.app/sandboxes
2026-03-26 18:04:15,373 INFO e2b.api.client_async: Response: 201 https://api.e2b.app/sandboxes
2026-03-26 18:04:15,374 INFO e2b.api: Response 201
2026-03-26 18:04:15,377 INFO e2b.api.client_async: Request: POST https://49999-iwslxvppi24raloknkq3c.e2b.app/execute
2026-03-26 18:04:15,506 INFO e2b.api.client_async: Response: 200 https://49999-iwslxvppi24raloknkq3c.e2b.app/execute
2026-03-26 18:04:15,618 INFO e2b.api: Request DELETE https://api.e2b.app/sandboxes/iwslxvppi24raloknkq3c
2026-03-26 18:04:15,619 INFO e2b.api.client_async: Request: DELETE https://api.e2b.app/sandboxes/iwslxvppi24raloknkq3c
2026-03-26 18:04:15,742 INFO e2b.api.client_async: Response: 204 https://api.e2b.app/sandboxes/iwslxvppi24raloknkq3c

Running fraud-004 ...


2026-03-26 18:13:04,918 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-26 18:13:04,923 INFO e2b.api: Request POST https://api.e2b.app/sandboxes
2026-03-26 18:13:04,924 INFO e2b.api.client_async: Request: POST https://api.e2b.app/sandboxes
2026-03-26 18:13:05,232 INFO e2b.api.client_async: Response: 201 https://api.e2b.app/sandboxes
2026-03-26 18:13:05,233 INFO e2b.api: Response 201
2026-03-26 18:13:05,236 INFO e2b.api.client_async: Request: POST https://49999-i6o0cklekoriqzcsisa1n.e2b.app/execute
2026-03-26 18:13:05,373 INFO e2b.api.client_async: Response: 200 https://49999-i6o0cklekoriqzcsisa1n.e2b.app/execute
2026-03-26 18:13:05,477 INFO e2b.api: Request DELETE https://api.e2b.app/sandboxes/i6o0cklekoriqzcsisa1n
2026-03-26 18:13:05,478 INFO e2b.api.client_async: Request: DELETE https://api.e2b.app/sandboxes/i6o0cklekoriqzcsisa1n
2026-03-26 18:13:05,595 INFO e2b.api.client_async: Response: 204 https://api.e2b.app/sandboxes/i6o0cklekoriqzcsisa1n

Running fraud-005 ...


2026-03-26 18:14:17,138 INFO google_adk.google.adk.models.google_llm: Response received from the model.
2026-03-26 18:14:17,144 INFO e2b.api: Request POST https://api.e2b.app/sandboxes
2026-03-26 18:14:17,144 INFO e2b.api.client_async: Request: POST https://api.e2b.app/sandboxes
2026-03-26 18:14:18,322 INFO e2b.api.client_async: Response: 201 https://api.e2b.app/sandboxes
2026-03-26 18:14:18,323 INFO e2b.api: Response 201
2026-03-26 18:14:18,326 INFO e2b.api.client_async: Request: POST https://49999-i6tkt1xuau43zrdedq9up.e2b.app/execute
2026-03-26 18:14:18,436 INFO e2b.api.client_async: Response: 200 https://49999-i6tkt1xuau43zrdedq9up.e2b.app/execute
2026-03-26 18:14:18,531 INFO e2b.api: Request DELETE https://api.e2b.app/sandboxes/i6tkt1xuau43zrdedq9up
2026-03-26 18:14:18,532 INFO e2b.api.client_async: Request: DELETE https://api.e2b.app/sandboxes/i6tkt1xuau43zrdedq9up
2026-03-26 18:14:33,945 INFO e2b.api.client_async: Response: 204 https://api.e2b.app/sandboxes/i6tkt1xuau43zrdedq9up

In [32]:
results_df = pd.DataFrame(results)
results_df[
    [
        "id",
        "structure_score",
        "signals_score",
        "evidence_score",
        "id_coverage_score",
        "predicted_transaction_id_count",
        "meets_min_transaction_ids",
        "ground_truth_score",
        "overall_score",
        "pass",
    ]
]

,id,structure_score,signals_score,evidence_score,id_coverage_score,predicted_transaction_id_count,meets_min_transaction_ids,ground_truth_score,overall_score,pass
0,fraud-001,1.0,1.0,0.75,0.0,0,False,0.0,0.55,False
1,fraud-002,1.0,1.0,1.00,0.0,0,False,0.0,0.60,False
2,fraud-003,1.0,1.0,0.75,0.0,0,False,0.0,0.55,False
3,fraud-004,1.0,1.0,1.00,0.0,0,False,0.0,0.60,False
4,fraud-005,1.0,1.0,1.00,1.0,4,True,0.0,0.80,True


## Aggregate Metrics and Failure Analysis

In [33]:
run_metrics = run_level_grader(results)
run_metrics_df = pd.DataFrame([run_metrics])
run_metrics_df

,cases,mean_overall_score,min_overall_score,max_overall_score,pass_rate,mean_ground_truth_score,mean_id_coverage_score,id_requirement_pass_rate
0,5,0.62,0.55,0.8,0.2,0.0,0.2,0.2


In [34]:
failed_cases_df = results_df.loc[~results_df["pass"], ["id", "prompt", "overall_score", "output"]]
failed_cases_df if not failed_cases_df.empty else pd.DataFrame([{"status": "All cases passed"}])

,id,prompt,overall_score,output
0,fraud-001,Identify the top 3 highest-risk clients in the...,0.55,ANALYSIS:\nBy querying the database to aggrega...
1,fraud-002,Find merchants that appear in more than one fr...,0.60,ANALYSIS:\nWe have identified multiple merchan...
2,fraud-003,Compare the risk profile of high-velocity clie...,0.55,ANALYSIS:\nA rigorous comparison between High-...
3,fraud-004,Write a fraud investigation brief covering all...,0.60,ANALYSIS:\nA comprehensive multi-signal fraud ...


## LLM-as-Judge Evaluation

While the deterministic graders check for keyword presence and structure, an LLM judge can assess deeper qualities like reasoning coherence, evidence quality, and whether recommendations are actionable.

`create_llm_as_judge_evaluator` returns an async evaluator backed by an OpenAI model. A custom rubric focuses scoring on fraud-relevant criteria.

In [27]:
from aieng.agent_evals.evaluation.graders import create_llm_as_judge_evaluator
from aieng.agent_evals.evaluation.graders.config import LLMRequestConfig

fraud_rubric = """
1. **signal_coverage** — Score 1 only if the output explicitly addresses ALL fraud signals mentioned in the input prompt \
(e.g. if the prompt asks about both high-value amounts AND velocity, both must be covered with specific findings). \
Score 0 if any requested signal is missing or only mentioned in passing without supporting detail.

2. **evidence_quality** — Score 1 only if the output cites at least 3 concrete data points \
(e.g. specific transaction amounts, client IDs, merchant names, counts, or dollar thresholds). \
Vague language like "high transaction counts" or "large amounts" without actual values does NOT count. \
Score 0 if fewer than 3 concrete data points are cited or if claims are stated without supporting figures.

3. **transaction_id_citation** — Score 1 only if the output includes at least 3 specific transaction IDs \
(e.g., "transaction_id: 12345, 12346, 12347"). Generic references to "transaction IDs" without listing actual numeric IDs do NOT qualify. \
Score 0 if fewer than 3 concrete transaction IDs are cited.

4. **actionability** — Score 1 only if the output includes at least 2 distinct, specific recommendations \
(e.g. "escalate client X for manual review", "flag merchant Y", "set alert threshold at $Z"). \
Generic phrases like "monitor activity" or "review transactions" without identifying a specific entity or threshold do NOT qualify. \
Score 0 if recommendations are absent, generic, or fewer than 2.

5. **reasoning_quality** — Score 1 only if the output explains WHY each finding is suspicious \
(e.g. links amounts to a threshold, compares against a baseline, or identifies a pattern). \
Score 0 if the output only lists findings without any causal or comparative reasoning.

6. **ground_truth_transaction_id_alignment** — Use `expected_output.ground_truth_positive_transaction_ids_sample` as the reference set. \
Score 1 only if at least 2 transaction IDs cited in the response are present in that ground-truth sample. \
Score 0 if fewer than 2 cited IDs match the ground-truth sample, or if no transaction IDs are cited.
"""

fraud_judge = create_llm_as_judge_evaluator(
    name="fraud_judge",
    rubric_markdown=fraud_rubric,
    model_config=LLMRequestConfig(temperature=0.0),
)

print("LLM judge created.")

LLM judge created.


In [35]:
async def run_llm_judge_case(case: dict, output_text: str) -> dict:
    """Run the LLM judge for a single eval case and return scores as a flat dict."""
    llm_evals = await fraud_judge(
        input={"prompt": case["prompt"]},
        output={"response": output_text},
        expected_output={
            "expected_signals": case["expected_signals"],
            "ground_truth_positive_transaction_ids_sample": case.get(
                "ground_truth_positive_transaction_ids_sample", []
            ),
            "ground_truth_positive_count": case.get("ground_truth_positive_count", 0),
        },
        metadata={"case_id": case["id"]},
    )
    return {e.name: e.value for e in llm_evals}


case_by_id = {case["id"]: case for case in eval_cases}

llm_results = []
for row in results:
    print(f"Judging {row['id']} ...")
    case_ref = case_by_id[row["id"]]
    llm_case = {
        "id": case_ref["id"],
        "prompt": case_ref["prompt"],
        "expected_signals": case_ref["expected_signals"],
        "ground_truth_positive_transaction_ids_sample": case_ref.get(
            "ground_truth_positive_transaction_ids", []
        )[:200],
        "ground_truth_positive_count": case_ref.get("ground_truth_positive_count", 0),
    }
    llm_scores = await run_llm_judge_case(case=llm_case, output_text=row["output"])
    llm_results.append({"id": row["id"], **llm_scores})

llm_results_df = pd.DataFrame(llm_results)
llm_results_df

Judging fraud-001 ...
Judging fraud-002 ...
Judging fraud-003 ...
Judging fraud-004 ...
Judging fraud-005 ...


,id,signal_coverage,evidence_quality,transaction_id_citation,actionability,reasoning_quality,ground_truth_transaction_id_alignment
0,fraud-001,1,1,1,1,1,0
1,fraud-002,1,1,1,1,1,0
2,fraud-003,1,1,1,1,1,0
3,fraud-004,1,1,1,1,1,0
4,fraud-005,1,1,1,1,1,0


In [36]:
# Merge LLM judge scores with deterministic scores for a side-by-side comparison
combined_df = results_df[["id", "structure_score", "signals_score", "evidence_score", "overall_score", "id_coverage_score", "pass"]].merge(
    llm_results_df, on="id"
)
combined_df

,id,structure_score,signals_score,evidence_score,overall_score,id_coverage_score,pass,signal_coverage,evidence_quality,transaction_id_citation,actionability,reasoning_quality,ground_truth_transaction_id_alignment
0,fraud-001,1.0,1.0,0.75,0.55,0.0,False,1,1,1,1,1,0
1,fraud-002,1.0,1.0,1.00,0.60,0.0,False,1,1,1,1,1,0
2,fraud-003,1.0,1.0,0.75,0.55,0.0,False,1,1,1,1,1,0
3,fraud-004,1.0,1.0,1.00,0.60,0.0,False,1,1,1,1,1,0
4,fraud-005,1.0,1.0,1.00,0.80,1.0,True,1,1,1,1,1,0
